# 05 — Comparação final: baseline × enriquecido
> **LAB-VET-01 — estudo com PaySim (dados sintéticos).** Não use dados confidenciais, credenciais ou identificadores reais neste repositório. Execute as células em ordem. Os resultados históricos estão documentados separadamente; as saídas foram removidas para publicação.

**Protocolo:** hiperparâmetros idênticos; escolha de capacidade operacional na validação; teste reservado avaliado uma vez. O teste tem prevalência diferente da validação. Este notebook reexecuta os treinamentos; pode consumir memória e tempo.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Coloque o CSV público PaySim em data/ (não versionar arquivos de dados).
DATA_PATH = Path("data/PS_20174392719_1491204439457_log.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset ausente: {DATA_PATH.resolve()}. Consulte o README.")
df = pd.read_csv(DATA_PATH)
print("Dataset:", df.shape)

In [ ]:
treino = df.loc[df["step"] <= 300].copy()
validacao = df.loc[(df["step"] > 300) & (df["step"] <= 400)].copy()
teste = df.loc[df["step"] > 400].copy()
print("Treino:", treino.shape, "Validação:", validacao.shape, "Teste:", teste.shape)

In [ ]:
features_baseline = ["amount", "oldbalanceOrg", "oldbalanceDest", "step", "type_CASH_IN", "type_CASH_OUT", "type_DEBIT", "type_PAYMENT", "type_TRANSFER"]
def montar_X(dados, colunas):
    return pd.get_dummies(dados[["amount", "oldbalanceOrg", "oldbalanceDest", "step", "type"]], columns=["type"], dtype="int8").reindex(columns=colunas, fill_value=0)

In [ ]:
from xgboost import XGBClassifier
def novo_modelo():
    return XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, objective="binary:logistic", eval_metric="aucpr", tree_method="hist", random_state=42, n_jobs=-1)

In [ ]:
def adicionar_features_historicas(dados):
    """Usa somente transações de steps estritamente anteriores; não utiliza rótulos."""
    base = dados.copy()
    agregado = (base.groupby(["nameDest", "step"], sort=True)
                .agg(qtd_step=("amount", "size"), volume_step=("amount", "sum"))
                .reset_index().sort_values(["nameDest", "step"]))
    grupo = agregado.groupby("nameDest", sort=False)
    agregado["destino_tx_anteriores"] = grupo["qtd_step"].cumsum() - agregado["qtd_step"]
    agregado["destino_volume_anterior"] = grupo["volume_step"].cumsum() - agregado["volume_step"]
    base = base.merge(agregado[["nameDest", "step", "destino_tx_anteriores", "destino_volume_anterior"]], on=["nameDest", "step"], how="left", validate="many_to_one", sort=False)
    base["destino_sem_historico"] = (base["destino_tx_anteriores"] == 0).astype("int8")
    assert len(base) == len(dados)
    assert base[["destino_tx_anteriores", "destino_volume_anterior", "destino_sem_historico"]].notna().all().all()
    return base

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

def avaliar(y, prob):
    pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"precision": precision_score(y, pred, zero_division=0), "recall": recall_score(y, pred, zero_division=0), "f1": f1_score(y, pred, zero_division=0), "average_precision": average_precision_score(y, prob), "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn)}

def avaliar_top_k(y, prob, k):
    labels = np.asarray(y)
    idx = np.argsort(-np.asarray(prob), kind="stable")[:k]
    tp = int(labels[idx].sum())
    return {"Top_K": k, "TP": tp, "FP": len(idx)-tp, "precision": tp/len(idx), "recall": tp/int(labels.sum())}

## Features cronológicas de treino e validação

In [ ]:
partes = pd.concat([treino.assign(_particao="treino"), validacao.assign(_particao="validacao")], ignore_index=True)
partes = adicionar_features_historicas(partes)
treino_grafo = partes.loc[partes["_particao"] == "treino"]
validacao_grafo = partes.loc[partes["_particao"] == "validacao"]
features_relacionais = ["destino_tx_anteriores", "destino_volume_anterior", "destino_sem_historico"]
features_modelo = features_baseline + features_relacionais

def montar_X_enriquecido(dados):
    return pd.get_dummies(dados[["amount", "oldbalanceOrg", "oldbalanceDest", "step", "type"] + features_relacionais], columns=["type"], dtype="int8").reindex(columns=features_modelo, fill_value=0)

## Treinar ambos os modelos sobre o mesmo treino

In [ ]:
y_treino = treino["isFraud"]
y_validacao = validacao["isFraud"]
modelo_baseline = novo_modelo()
modelo_enriquecido = novo_modelo()
modelo_baseline.fit(montar_X(treino, features_baseline), y_treino)
modelo_enriquecido.fit(montar_X_enriquecido(treino_grafo), y_treino)
proba_baseline = modelo_baseline.predict_proba(montar_X(validacao, features_baseline))[:, 1]
proba_enriquecido = modelo_enriquecido.predict_proba(montar_X_enriquecido(validacao_grafo))[:, 1]
print(pd.DataFrame({"Baseline": avaliar(y_validacao, proba_baseline), "Enriquecido": avaliar(y_validacao, proba_enriquecido)}))
print(pd.DataFrame([{"Top_K": k, "TP_baseline": avaliar_top_k(y_validacao, proba_baseline, k)["TP"], "TP_enriquecido": avaliar_top_k(y_validacao, proba_enriquecido, k)["TP"]} for k in [250,500,750,1000,1250]]))

## Teste reservado — sem ajustar o modelo com base nos resultados

In [ ]:
historico_completo = adicionar_features_historicas(pd.concat([treino.assign(_particao="treino"), validacao.assign(_particao="validacao"), teste.assign(_particao="teste")], ignore_index=True))
teste_grafo = historico_completo.loc[historico_completo["_particao"] == "teste"]
y_teste = teste["isFraud"]
prob_teste_baseline = modelo_baseline.predict_proba(montar_X(teste, features_baseline))[:, 1]
prob_teste_enriquecido = modelo_enriquecido.predict_proba(montar_X_enriquecido(teste_grafo))[:, 1]
print(pd.DataFrame({"Baseline": avaliar(y_teste, prob_teste_baseline), "Enriquecido": avaliar(y_teste, prob_teste_enriquecido)}))
print(pd.DataFrame([{"Top_K": k, "TP_baseline": avaliar_top_k(y_teste, prob_teste_baseline, k)["TP"], "TP_enriquecido": avaliar_top_k(y_teste, prob_teste_enriquecido, k)["TP"]} for k in [250,500,750,1000,1250]]))

## Conclusão
Na execução original, o enriquecimento não apresentou ganho consistente no teste. A comparação demonstra features históricas relacionais, **não** superioridade exclusiva do Neo4j. Os números devem ser reproduzidos antes de qualquer afirmação adicional.

---
**Autoria**

```text
Alexandre Lotti
<!LOTTI>
```
